# OpenIRM — Exploratory Data Analysis (EDA)

This notebook performs exploratory analysis on the preprocessed CERT Insider Threat Dataset r4.2 daily feature matrix (`activity_features.parquet`).

### Objectives:
1. **Class Balance**: Evaluate the proportion of benign vs malicious user-day records.
2. **Activity Volume**: Inspect distribution of logon, file, email, and web activity across users.
3. **Time Distribution**: Compare normal business hours vs after-hours / weekend activity.
4. **Scenario Window Sanity Check**: Confirm zero truncation of malicious scenario timelines.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

parquet_path = Path('../backend/data/filtered/processed/activity_features.parquet')
df = pd.read_parquet(parquet_path)
print(f'Dataset Shape: {df.shape}')
df.head()

## 1. Class Balance Analysis

In [ ]:
counts = df['is_malicious'].value_counts()
print(counts)
print(f'Malicious Percentage: {df["is_malicious"].mean() * 100:.2f}%')

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=['Benign (0)', 'Malicious (1)'], y=[counts[0], counts[1]], palette=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('Class Balance: Daily User Activity Records')
plt.show()

## 2. Activity Volume per User

In [ ]:
user_totals = df.groupby('user')[['logon_count', 'file_count', 'email_count', 'web_visit_count']].sum()
user_totals.describe()

## 3. Time Distribution: Business Hours vs Off-Hours

In [ ]:
total_logons = df['logon_count'].sum()
after_hours_logons = df['logon_after_hours'].sum()
print(f'Total Logons: {int(total_logons):,}')
print(f'After Hours / Weekend Logons: {int(after_hours_logons):,} ({after_hours_logons / total_logons * 100:.2f}%)')

## 4. Malicious Scenario Window Sanity Check

In [ ]:
df_mal = df[df['is_malicious'] == 1]
print(f'Unique Malicious Users: {df_mal["user"].nunique()}')
print(f'Observation Start Date: {df["date_day"].min()}')
print(f'Observation End Date: {df["date_day"].max()}')